In [1]:
import pyspark
from pyspark.sql import SparkSession

# Initialize Spark Session with MinIO (S3a) AND Delta configurations
spark = SparkSession.builder \
    .appName("Z5008_A5_Verification") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# Verify Spark Version (Must be 3.5+)
spark_version = spark.version
print(f"Spark Version: {spark_version}")

# Create a test DataFrame to verify Spark functionality
data = [("Zanzibar", 100), ("Madras", 200), ("A5_Task", 300)]
df = spark.createDataFrame(data, ["Location", "Score"])

print("\n--- Testing Spark DataFrame ---")
df.show()

# Verify MinIO write/read functionality
test_path = "s3a://test-bucket/verification_data"

try:
    df.write.mode("overwrite").parquet(test_path)
    print("\n--- MinIO Write Successful ---")
    
    read_df = spark.read.parquet(test_path)
    print("\n--- MinIO Read Successful ---")
    read_df.show()
except Exception as e:
    print(f"\nMinIO Check Note: Ensure you have created a bucket named 'test-bucket' in the MinIO UI (http://localhost:9001). Error details: {e}")

print("Environment setup and verification complete!")

Spark Version: 3.5.0

--- Testing Spark DataFrame ---
+--------+-----+
|Location|Score|
+--------+-----+
|Zanzibar|  100|
|  Madras|  200|
| A5_Task|  300|
+--------+-----+


MinIO Check Note: Ensure you have created a bucket named 'test-bucket' in the MinIO UI (http://localhost:9001). Error details: An error occurred while calling o72.parquet.
: java.nio.file.AccessDeniedException: s3a://test-bucket/verification_data: getFileStatus on s3a://test-bucket/verification_data: com.amazonaws.services.s3.model.AmazonS3Exception: Forbidden (Service: Amazon S3; Status Code: 403; Error Code: 403 Forbidden; Request ID: 18AA0343577CF2CA; S3 Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8; Proxy: null), S3 Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8:403 Forbidden
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:255)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Generate 365 days of synthetic sales data for 3 different regions
df = spark.sql("""
    SELECT 
        EXPLODE(SEQUENCE(CAST('2023-01-01' AS DATE), CAST('2023-12-31' AS DATE), INTERVAL 1 DAY)) AS date
""")

regions = spark.createDataFrame([("North",), ("South",), ("East",)], ["region"])

# Cross join to get every date for every region, and add a randomized sales metric
dataset_df = df.crossJoin(regions) \
    .withColumn("sales", F.round(F.rand(seed=42) * 1000 + 500, 2))

print("Base Dataset (First 5 rows):")
dataset_df.show(5)

Base Dataset (First 5 rows):
+----------+------+-------+
|      date|region|  sales|
+----------+------+-------+
|2023-01-01| North|1301.75|
|2023-01-02| North|1156.56|
|2023-01-03| North| 751.56|
|2023-01-04| North| 707.34|
|2023-01-05| North|1139.29|
+----------+------+-------+
only showing top 5 rows



In [3]:
# Define the window specification: Partition by group, order by date, 6 previous days + current day
window_7_days = Window.partitionBy("region").orderBy("date").rowsBetween(-6, Window.currentRow)

rolling_avg_df = dataset_df.withColumn(
    "7_day_rolling_avg", 
    F.round(F.avg("sales").over(window_7_days), 2)
)

print("7-Day Rolling Average (Sorted by region and date):")
rolling_avg_df.orderBy("region", "date").show(10)

7-Day Rolling Average (Sorted by region and date):
+----------+------+-------+-----------------+
|      date|region|  sales|7_day_rolling_avg|
+----------+------+-------+-----------------+
|2023-01-01|  East|1297.13|          1297.13|
|2023-01-02|  East| 742.26|           1019.7|
|2023-01-03|  East|1209.74|          1083.04|
|2023-01-04|  East| 950.94|          1050.02|
|2023-01-05|  East| 803.58|          1000.73|
|2023-01-06|  East| 1079.4|          1013.84|
|2023-01-07|  East|1280.07|          1051.87|
|2023-01-08|  East| 557.39|            946.2|
|2023-01-09|  East|1320.93|          1028.86|
|2023-01-10|  East| 792.33|           969.23|
+----------+------+-------+-----------------+
only showing top 10 rows



In [4]:
# First, aggregate daily data to monthly data
monthly_df = dataset_df.withColumn("month", F.date_trunc("month", F.col("date"))) \
    .groupBy("region", "month") \
    .agg(F.round(F.sum("sales"), 2).alias("current_value"))

# Define window to look at previous month
window_mom = Window.partitionBy("region").orderBy("month")

# Compute previous value and growth percentage
mom_growth_df = monthly_df.withColumn(
    "previous_value", 
    F.lag("current_value", 1).over(window_mom)
).withColumn(
    "growth_percentage",
    F.when(F.col("previous_value").isNull(), "N/A (First Month)") \
     .otherwise(F.concat(F.round(((F.col("current_value") - F.col("previous_value")) / F.col("previous_value")) * 100, 2).cast("string"), F.lit("%")))
)

print("Month-over-Month Growth Rate:")
mom_growth_df.orderBy("region", "month").show(15)

Month-over-Month Growth Rate:
+------+-------------------+-------------+--------------+-----------------+
|region|              month|current_value|previous_value|growth_percentage|
+------+-------------------+-------------+--------------+-----------------+
|  East|2023-01-01 00:00:00|     30300.96|          NULL|N/A (First Month)|
|  East|2023-02-01 00:00:00|     30977.27|      30300.96|            2.23%|
|  East|2023-03-01 00:00:00|     29106.34|      30977.27|           -6.04%|
|  East|2023-04-01 00:00:00|     29889.99|      29106.34|            2.69%|
|  East|2023-05-01 00:00:00|     32997.98|      29889.99|            10.4%|
|  East|2023-06-01 00:00:00|     29768.84|      32997.98|           -9.79%|
|  East|2023-07-01 00:00:00|     29408.63|      29768.84|           -1.21%|
|  East|2023-08-01 00:00:00|     27754.76|      29408.63|           -5.62%|
|  East|2023-09-01 00:00:00|     26380.22|      27754.76|           -4.95%|
|  East|2023-10-01 00:00:00|     31470.58|      26380.22| 

In [5]:
# Add month column to base dataset for this specific query
dataset_with_month = dataset_df.withColumn("month", F.date_format(F.col("date"), "yyyy-MM"))

dataset_with_month.createOrReplaceTempView("sales_data")

# Using a CTE/subquery since QUALIFY isn't supported in vanilla Spark 3.5
top_3_df = spark.sql("""
    WITH RankedSales AS (
        SELECT 
            region, 
            month, 
            date, 
            sales,
            RANK() OVER (PARTITION BY region, month ORDER BY sales DESC) as rnk
        FROM sales_data
    )
    SELECT *
    FROM RankedSales
    WHERE rnk <= 3
    ORDER BY region, month, rnk
""")

print("Top 3 Sales Records per Region per Month:")
top_3_df.show(15)

Top 3 Sales Records per Region per Month:
+------+-------+----------+-------+---+
|region|  month|      date|  sales|rnk|
+------+-------+----------+-------+---+
|  East|2023-01|2023-01-23|1392.42|  1|
|  East|2023-01|2023-01-24| 1387.8|  2|
|  East|2023-01|2023-01-09|1320.93|  3|
|  East|2023-02|2023-02-25|1487.43|  1|
|  East|2023-02|2023-02-01|1411.16|  2|
|  East|2023-02|2023-02-02|1393.94|  3|
|  East|2023-03|2023-03-04|1461.78|  1|
|  East|2023-03|2023-03-06|1422.39|  2|
|  East|2023-03|2023-03-14|1394.61|  3|
|  East|2023-04|2023-04-22|1498.58|  1|
|  East|2023-04|2023-04-29|1477.75|  2|
|  East|2023-04|2023-04-12|1462.21|  3|
|  East|2023-05|2023-05-04|1481.49|  1|
|  East|2023-05|2023-05-27|1426.31|  2|
|  East|2023-05|2023-05-19|1414.22|  3|
+------+-------+----------+-------+---+
only showing top 15 rows



In [6]:
# Define window covering the entire partition
window_full = Window.partitionBy("region") \
    .orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

# Get first and last values, calculate change, and distinct to get one row per region
change_df = dataset_df.withColumn("first_value", F.first("sales").over(window_full)) \
    .withColumn("last_value", F.last("sales").over(window_full)) \
    .withColumn("total_change", F.round(F.col("last_value") - F.col("first_value"), 2)) \
    .select("region", "first_value", "last_value", "total_change") \
    .distinct()

print("Total Change (First vs Last Value):")
change_df.orderBy("region").show()

Total Change (First vs Last Value):
+------+-----------+----------+------------+
|region|first_value|last_value|total_change|
+------+-----------+----------+------------+
|  East|    1297.13|   1066.86|     -230.27|
| North|    1301.75|   1244.95|       -56.8|
| South|    1298.53|    853.45|     -445.08|
+------+-----------+----------+------------+



In [7]:
import time
from pyspark.sql import functions as F

# Generate 500,000 rows for Sales
df_sales = spark.range(0, 500000).withColumn("store_id", (F.rand(seed=42) * 500).cast("int")) \
    .withColumn("product_category", (F.rand(seed=42) * 10).cast("int")) \
    .withColumn("amount", F.round(F.rand(seed=42) * 100, 2))

# Generate 500 rows for Stores
df_stores = spark.range(0, 500).withColumn("store_id", F.col("id")) \
    .withColumn("region", F.concat(F.lit("Region_"), (F.rand(seed=42) * 5).cast("int")))

# Register as temp views
df_sales.createOrReplaceTempView("sales")
df_stores.createOrReplaceTempView("stores")

# Define our intentional slow query
slow_query = """
    SELECT 
        st.region, 
        sa.product_category, 
        st.store_id,
        SUM(sa.amount) as total_sales
    FROM sales sa
    JOIN stores st ON sa.store_id = st.store_id
    GROUP BY st.region, sa.product_category, st.store_id
    ORDER BY total_sales DESC
"""

In [8]:
# Disable optimizations for baseline
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

start_time = time.time()
baseline_df = spark.sql(slow_query)
baseline_df.collect() # Action to trigger execution
baseline_time = time.time() - start_time

print(f"Baseline Time: {baseline_time:.2f} seconds")

Baseline Time: 5.73 seconds


In [9]:
# Optimization A: Reduce shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "10")

start_time = time.time()
opt_a_df = spark.sql(slow_query)
opt_a_df.collect()
time_a = time.time() - start_time

print(f"Time After Opt A: {time_a:.2f} seconds")

Time After Opt A: 1.05 seconds


In [10]:
print("--- EXPLAIN PLAN BEFORE BROADCAST (SortMergeJoin) ---")
opt_a_df.explain()

# Optimization B: Broadcast the small table
opt_b_df = df_sales.join(F.broadcast(df_stores), "store_id") \
    .groupBy("region", "product_category", "store_id") \
    .agg(F.sum("amount").alias("total_sales")) \
    .orderBy(F.col("total_sales").desc())

print("\n--- EXPLAIN PLAN AFTER BROADCAST (BroadcastHashJoin) ---")
opt_b_df.explain()

start_time = time.time()
opt_b_df.collect()
time_b = time.time() - start_time

print(f"\nTime After Opt B: {time_b:.2f} seconds")

--- EXPLAIN PLAN BEFORE BROADCAST (SortMergeJoin) ---
== Physical Plan ==
*(6) Sort [total_sales#253 DESC NULLS LAST], true, 0
+- Exchange rangepartitioning(total_sales#253 DESC NULLS LAST, 10), ENSURE_REQUIREMENTS, [plan_id=996]
   +- *(5) HashAggregate(keys=[region#238, product_category#224, store_id#235L], functions=[sum(amount#228)])
      +- *(5) HashAggregate(keys=[region#238, product_category#224, store_id#235L], functions=[partial_sum(amount#228)])
         +- *(5) Project [product_category#224, amount#228, store_id#235L, region#238]
            +- *(5) SortMergeJoin [cast(store_id#221 as bigint)], [store_id#235L], Inner
               :- *(2) Sort [cast(store_id#221 as bigint) ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(cast(store_id#221 as bigint), 10), ENSURE_REQUIREMENTS, [plan_id=980]
               :     +- *(1) Filter isnotnull(store_id#221)
               :        +- *(1) Project [store_id#221, product_category#224, round((rand(42) * 100.0)

In [11]:
# Reset manual optimizations
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760") # 10MB default

# Optimization C: Enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")

start_time = time.time()
opt_c_df = spark.sql(slow_query)
opt_c_df.collect()
time_c = time.time() - start_time

print(f"Time After Opt C (AQE): {time_c:.2f} seconds")

Time After Opt C (AQE): 0.95 seconds


In [12]:
# Optimization D: Cache the large DataFrame
df_sales.cache()
df_sales.count() # Force materialization

start_time = time.time()
opt_d_df = spark.sql(slow_query)
opt_d_df.collect()
time_d = time.time() - start_time

print(f"Time After Opt D (Cache): {time_d:.2f} seconds")

# Ensure AQE + Cache + Partitioning are all active for the absolute best time
spark.conf.set("spark.sql.shuffle.partitions", "10")
start_time = time.time()
final_df = spark.sql(slow_query)
final_df.collect()
time_final = time.time() - start_time

# Print Final Summary Table
print("\n" + "="*50)
print(f"{'Optimization Strategy':<30} | {'Execution Time':<15}")
print("="*50)
print(f"{'Baseline (No Opts)':<30} | {baseline_time:.2f} s")
print(f"{'After A (Partitions=10)':<30} | {time_a:.2f} s")
print(f"{'After B (Broadcast)':<30} | {time_b:.2f} s")
print(f"{'After C (AQE Only)':<30} | {time_c:.2f} s")
print(f"{'After D (Cache)':<30} | {time_d:.2f} s")
print(f"{'After A + B + C + D':<30} | {time_final:.2f} s")
print("="*50)

speedup = baseline_time / time_final
print(f"Final Cumulative Speedup vs Baseline: {speedup:.2f}x")

Time After Opt D (Cache): 0.84 seconds

Optimization Strategy          | Execution Time 
Baseline (No Opts)             | 5.73 s
After A (Partitions=10)        | 1.05 s
After B (Broadcast)            | 0.93 s
After C (AQE Only)             | 0.95 s
After D (Cache)                | 0.84 s
After A + B + C + D            | 0.47 s
Final Cumulative Speedup vs Baseline: 12.26x


In [13]:
import time
from pyspark.sql import functions as F

# Generate skewed data: 90% of rows get key '1', 10% get a random key from 2 to 10
skewed_df = spark.range(0, 2000000).withColumn(
    "join_key",
    F.when(F.rand(seed=42) < 0.9, F.lit(1)).otherwise((F.rand(seed=42) * 9 + 2).cast("int"))
).withColumn("value", F.round(F.rand(seed=42) * 100, 2))

# Task 3a Baseline Aggregation
print("Running Baseline Skewed Aggregation...")
start_time = time.time()
skewed_df.groupBy("join_key").agg(F.sum("value").alias("total_value")).collect()
baseline_skew_time = time.time() - start_time
print(f"Baseline Execution Time: {baseline_skew_time:.2f} seconds")

Running Baseline Skewed Aggregation...
Baseline Execution Time: 0.66 seconds


In [14]:
SALT_FACTOR = 20

# Step (i) Add a random salt column
salted_df = skewed_df.withColumn("salt", (F.rand(seed=42) * SALT_FACTOR).cast("int")) \
    .withColumn("salted_key", F.concat(F.col("join_key"), F.lit("_"), F.col("salt")))

start_time = time.time()

# Step (ii) Compute partial aggregation on the salted key
partial_agg = salted_df.groupBy("join_key", "salted_key").agg(F.sum("value").alias("partial_sum"))

# Step (iii) Compute final aggregation after removing the salt (grouping by original key)
final_agg = partial_agg.groupBy("join_key").agg(F.sum("partial_sum").alias("total_value"))
final_agg.collect()

salted_time = time.time() - start_time
print(f"Salted Execution Time: {salted_time:.2f} seconds")

Salted Execution Time: 1.00 seconds


Task 3d: Salting vs. AQE Skew Join
Salting is a manual engineering technique that requires modifying the actual dataframe logic, adding complexity to the codebase. It is primarily preferred when dealing with skewed aggregations (like large groupBy operations) where no join is involved, or when using older versions of Spark (< 3.0) that lack modern automated features. In contrast, AQE Skew Join handling is highly automated and evaluates data statistics dynamically at runtime to split skewed partitions seamlessly. However, the trade-off is that AQE's skew handling only triggers during specific join operations (like SortMergeJoins) and requires careful tuning of configuration thresholds to ensure it activates correctly without causing unnecessary overhead on smaller datasets.

Task 4a: Bucketing SetupCreate a Markdown Cell and paste:Task 4a: Bucketing Setup
We are generating a 50,000-row orders dataset and a smaller lookup dataset. We write them out first as standard Parquet files, and then as bucketed tables with $N=8$ buckets on the customer_id join key. Finally, we use DESCRIBE EXTENDED to verify that the bucket metadata is successfully stored

In [ ]:
import time
from pyspark.sql import functions as F

# Disable optimizations so Spark is forced to do a SortMergeJoin 
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")

# 1. Generate the DataFrames
orders_df = spark.range(0, 50000).withColumn("customer_id", (F.rand(seed=42) * 5000).cast("int")) \
    .withColumn("order_amount", F.round(F.rand(seed=42) * 100, 2))

lookup_df = spark.range(0, 5000).withColumn("customer_id", F.col("id").cast("int")) \
    .withColumn("customer_segment", F.when(F.rand() > 0.5, "Premium").otherwise("Standard"))

# 2. Write Unbucketed Data directly to local Jupyter disk!
path_unbucketed_orders = "/home/jovyan/work/unbucketed_orders"
path_unbucketed_lookup = "/home/jovyan/work/unbucketed_lookup"

orders_df.write.mode("overwrite").parquet(path_unbucketed_orders)
lookup_df.write.mode("overwrite").parquet(path_unbucketed_lookup)

# 3. Write Bucketed Tables (defaults to local spark-warehouse)
orders_df.write.mode("overwrite").bucketBy(8, "customer_id").saveAsTable("bucketed_orders")
lookup_df.write.mode("overwrite").bucketBy(8, "customer_id").saveAsTable("bucketed_lookup")

# 4. Verify Bucket Metadata
print("--- Verifying Bucketed Orders Metadata ---")
spark.sql("DESCRIBE EXTENDED bucketed_orders").filter(F.col("col_name").isin("Num Buckets", "Bucket Columns")).show(truncate=False)

Task 4b: Non-Bucketed Join
When joining the unbucketed parquet files, Spark has to read the data and perform a physical shuffle across the network to ensure matching customer_id keys end up on the same worker node. We can see this in the execution plan as Exchange hashpartitioning nodes

In [18]:
# Read unbucketed data
unbucketed_orders = spark.read.parquet("s3a://test-bucket/unbucketed_orders")
unbucketed_lookup = spark.read.parquet("s3a://test-bucket/unbucketed_lookup")

unbucketed_join = unbucketed_orders.join(unbucketed_lookup, "customer_id") \
    .groupBy("customer_segment").agg(F.sum("order_amount"))

print("--- EXPLAIN PLAN: UNBUCKETED (Notice the Exchange nodes) ---")
unbucketed_join.explain(mode="formatted")

start_time = time.time()
unbucketed_join.collect()
unbucketed_time = time.time() - start_time

print(f"\nExecution Time (Unbucketed): {unbucketed_time:.2f} seconds")

Py4JJavaError: An error occurred while calling o417.parquet.
: java.nio.file.AccessDeniedException: s3a://test-bucket/unbucketed_orders: getFileStatus on s3a://test-bucket/unbucketed_orders: com.amazonaws.services.s3.model.AmazonS3Exception: Forbidden (Service: Amazon S3; Status Code: 403; Error Code: 403 Forbidden; Request ID: 18AA036D526C2D72; S3 Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8; Proxy: null), S3 Extended Request ID: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8:403 Forbidden
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:255)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3796)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$exists$34(S3AFileSystem.java:4703)
	at org.apache.hadoop.fs.statistics.i

Task 4c: Bucketed Join Experiment
By reading directly from our explicitly bucketed tables, Spark knows the data is already partitioned by customer_id into 8 buckets on disk. It completely skips the shuffle phase, resulting in an execution plan with fewer Exchange nodes and faster execution.

In [ ]:
# Read bucketed tables
bucketed_orders = spark.table("bucketed_orders")
bucketed_lookup = spark.table("bucketed_lookup")

bucketed_join = bucketed_orders.join(bucketed_lookup, "customer_id") \
    .groupBy("customer_segment").agg(F.sum("order_amount"))

print("--- EXPLAIN PLAN: BUCKETED (Exchange nodes eliminated for the join) ---")
bucketed_join.explain(mode="formatted")

start_time = time.time()
bucketed_join.collect()
bucketed_time = time.time() - start_time

print(f"\nExecution Time (Bucketed): {bucketed_time:.2f} seconds")

# Report Speedup
if bucketed_time > 0:
    speedup = unbucketed_time / bucketed_time
    print(f"Bucketing Speedup: {speedup:.2f}x")
else:
    print("Bucketing Speedup: Exceptionally fast (near zero seconds)!")

Task 4d: Why matching buckets matter
Bucketing only eliminates shuffles when both tables share the exact same number of buckets on the same join column, because it guarantees that matching keys from both tables inherently reside in the correspondingly numbered file buckets, allowing Spark to join them locally without any cross-node network transfers

Task 5: Reflectiona. Repartition vs. Coalesce
You should use repartition(N) when you need to increase the number of partitions or guarantee perfectly uniform data distribution across nodes, which inherently requires a full network shuffle. A concrete production scenario for repartition(N) is preparing a heavily filtered dataset before writing it to distributed storage, ensuring you generate uniformly sized Parquet files rather than thousands of skewed, tiny ones. Conversely, coalesce(N) should be used exclusively to decrease the number of partitions, as it avoids a full shuffle by safely combining existing local partitions on the same worker node. A classic production scenario for coalesce is collapsing 10,000 tiny partitions down to 100 after a massive filter operation right before writing to disk. The performance implications of choosing the wrong one are severe: using repartition to simply reduce partitions wastes massive compute resources on unnecessary network shuffles, while using coalesce on highly skewed data can accidentally overload a single executor and trigger an Out-Of-Memory (OOM) crash.b. AQE Features and Limitations
AQE's "partition coalescing" automatically combines multiple tiny, post-shuffle partitions into fewer, optimally sized partitions at runtime, saving Spark the overhead of spinning up unnecessary tasks. The "dynamic join strategy switch" allows Spark to change its execution plan mid-flight; if it planned a heavy SortMergeJoin but realizes after filtering that a table is actually small enough, it will instantly convert it to a blazing-fast BroadcastHashJoin. However, AQE might not help if your performance bottleneck occurs during the initial data read before any shuffles happen, or if your dataset contains extreme skew that exceeds AQE's configured splitting thresholds. Under those specific circumstances, you would need to apply manual tuning techniques instead. Examples of manual tuning include using salted keys to physically break up the skew, implementing bucketing on the storage layer, or explicitly applying broadcast() hints in your PySpark code.